# ❄️ NYC Snow Map — Data Pipeline### Winter 2025–2026This notebook builds all data files needed for the interactive NYC neighborhood-level snowfall visualization.**Outputs:**- `nyc_neighborhoods.geojson` — ~200 NTAs with snowfall properties- `nyc_boroughs.geojson` — 5 boroughs with aggregated totals- `nyc_storms.json` — individual storm events- `nyc_historical.json` — historical season totals (Central Park, 1869–present)**Architecture:** Neighborhood Tabulation Areas (NTAs) instead of street segments. Borough-level default view, neighborhood drill-down on zoom.

In [1]:
# CELL 1 — Install dependencies
!pip install geopandas scipy shapely requests pandas numpy CoCoRaHS-Download-Tool -q

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [2]:
# CELL 2 — Imports
import os, json, requests, time, glob
import pandas as pd
import numpy as np
import geopandas as gpd
from scipy.spatial import cKDTree
from shapely.geometry import Point
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)
print('Imports ready')

Imports ready


In [3]:
# CELL 3 — CONFIGURATION  ← paste your NOAA key here
NOAA_API_KEY   = 'NDJeXTCmRuSQrLzNQVPNZhvDLsMrAytO'

# NYC bounding box (all 5 boroughs)
NYC_BBOX = dict(lat_min=40.49, lat_max=40.92, lon_min=-74.26, lon_max=-73.70)

# Central Park — the canonical NYC weather station
STATION_ID     = 'GHCND:USW00094728'
STATION_NAME   = 'Central Park, NY'

SEASON_START = '2025-10-01'
SEASON_END   = '2026-03-31'
IDW_POWER    = 2

# CoCoRaHS config — fetched automatically via pip package
COCORAHS_STATE = 'NY'
COCORAHS_START = '2025-10-01'
COCORAHS_END   = '2026-03-31'

print('Config set — NYC mode')
print(f'Station: {STATION_NAME} ({STATION_ID})')
print(f'Bbox: {NYC_BBOX}')

Config set — NYC mode
Station: Central Park, NY (GHCND:USW00094728)
Bbox: {'lat_min': 40.49, 'lat_max': 40.92, 'lon_min': -74.26, 'lon_max': -73.7}


## 1. Fetch CoCoRaHS Data (NYC Area)
Automated via `CoCoRaHS-Download-Tool` pip package — no manual CSV download needed.

In [4]:
# CELL 4 — Fetch CoCoRaHS daily reports via CoCoRaHS-Download-Tool
import subprocess, io

print(f'Fetching CoCoRaHS data for state={COCORAHS_STATE}, {COCORAHS_START} to {COCORAHS_END}...')

try:
    result = subprocess.run(
        ['python', '-m', 'cocorahs_dlt',
         '--state', COCORAHS_STATE,
         'data',
         '--start_date', COCORAHS_START,
         '--end_date', COCORAHS_END,
         '--format', 'CSV'],
        capture_output=True, text=True, timeout=120
    )

    if result.returncode == 0 and len(result.stdout) > 100:
        df_raw = pd.read_csv(io.StringIO(result.stdout))
        print(f'CoCoRaHS fetched: {len(df_raw):,} rows')
    else:
        print(f'CoCoRaHS CLI returned no data or error.')
        print(f'stderr: {result.stderr[:500]}')
        df_raw = None
except Exception as e:
    print(f'CoCoRaHS fetch failed: {e}')
    df_raw = None

# If CLI didn't work, try direct URL approach as fallback
if df_raw is None:
    print('\nTrying direct CoCoRaHS web export as fallback...')
    try:
        url = (f'https://data.cocorahs.org/cocorahs/export/exportreports.aspx'
               f'?ReportType=Daily&Format=CSV&State={COCORAHS_STATE}'
               f'&StartDate={COCORAHS_START}&EndDate={COCORAHS_END}')
        r = requests.get(url, timeout=60)
        if r.status_code == 200 and len(r.text) > 100:
            df_raw = pd.read_csv(io.StringIO(r.text))
            print(f'CoCoRaHS web export: {len(df_raw):,} rows')
        else:
            print('Web export also failed — continuing without CoCoRaHS')
    except Exception as e:
        print(f'Web export failed: {e}')

if df_raw is not None:
    df_raw.columns = df_raw.columns.str.strip()
    df_raw = df_raw.apply(lambda c: c.str.strip() if c.dtype == 'object' else c)

    # Detect column names (may vary between CLI and web export)
    date_col = next((c for c in df_raw.columns if 'date' in c.lower() and 'obs' in c.lower()),
                    next((c for c in df_raw.columns if 'date' in c.lower()), None))
    lat_col  = next((c for c in df_raw.columns if 'lat' in c.lower()), None)
    lon_col  = next((c for c in df_raw.columns if 'lon' in c.lower()), None)
    snow_col = next((c for c in df_raw.columns if 'snow' in c.lower() and 'new' in c.lower()),
                    next((c for c in df_raw.columns if 'snow' in c.lower()), None))
    stn_col  = next((c for c in df_raw.columns if 'station' in c.lower() and 'num' in c.lower()),
                    next((c for c in df_raw.columns if 'station' in c.lower()), None))
    name_col_c = next((c for c in df_raw.columns if 'station' in c.lower() and 'name' in c.lower()), stn_col)

    print(f'Detected columns: date={date_col}, lat={lat_col}, lon={lon_col}, snow={snow_col}, station={stn_col}')

    # Rename to standard names
    rename_map = {}
    if date_col and date_col != 'ObservationDate': rename_map[date_col] = 'ObservationDate'
    if lat_col and lat_col != 'Latitude': rename_map[lat_col] = 'Latitude'
    if lon_col and lon_col != 'Longitude': rename_map[lon_col] = 'Longitude'
    if snow_col and snow_col != 'NewSnowDepth': rename_map[snow_col] = 'NewSnowDepth'
    if stn_col and stn_col != 'StationNumber': rename_map[stn_col] = 'StationNumber'
    if name_col_c and name_col_c != 'StationName': rename_map[name_col_c] = 'StationName'
    df_raw = df_raw.rename(columns=rename_map)

    df_raw['ObservationDate'] = pd.to_datetime(df_raw['ObservationDate'])
    df_raw['Latitude']        = pd.to_numeric(df_raw['Latitude'],        errors='coerce')
    df_raw['Longitude']       = pd.to_numeric(df_raw['Longitude'],       errors='coerce')
    df_raw['NewSnowDepth']    = pd.to_numeric(df_raw['NewSnowDepth'],    errors='coerce').fillna(0)

    df = df_raw[(df_raw['ObservationDate'] >= SEASON_START) &
                (df_raw['ObservationDate'] <= SEASON_END)].copy()

    # Filter to NYC bounding box
    mask = (df['Latitude'].between(NYC_BBOX['lat_min'], NYC_BBOX['lat_max']) &
            df['Longitude'].between(NYC_BBOX['lon_min'], NYC_BBOX['lon_max']))
    df_nyc = df[mask].copy()

    print(f'\nTotal NY records : {len(df):,}')
    print(f'NYC bbox records : {len(df_nyc):,}')
    print(f'NYC stations     : {df_nyc["StationNumber"].nunique()}')
    if len(df_nyc) > 0:
        print(f'Date range       : {df_nyc["ObservationDate"].min().date()} to {df_nyc["ObservationDate"].max().date()}')
else:
    df_nyc = pd.DataFrame(columns=['StationNumber','StationName','Latitude','Longitude',
                                    'ObservationDate','NewSnowDepth'])
    print('No CoCoRaHS data — proceeding with NOAA Central Park only')

Fetching CoCoRaHS data for state=NY, 2025-10-01 to 2026-03-31...
CoCoRaHS CLI returned no data or error.
stderr: /usr/bin/python3: No module named cocorahs_dlt


Trying direct CoCoRaHS web export as fallback...
CoCoRaHS web export: 55,363 rows
Detected columns: date=ObservationDate, lat=Latitude, lon=Longitude, snow=NewSnowDepth, station=StationNumber

Total NY records : 55,363
NYC bbox records : 712
NYC stations     : 11
Date range       : 2025-10-01 to 2026-03-11


In [5]:
# CELL 5 — Seasonal totals per CoCoRaHS station (if available)
if len(df_nyc) > 0:
    station_totals = (
        df_nyc
        .groupby(['StationNumber','StationName','Latitude','Longitude'])['NewSnowDepth']
        .sum().reset_index()
        .rename(columns={'NewSnowDepth':'seasonal_snow_in'})
    )
    counts = df_nyc.groupby('StationNumber').size().reset_index(name='ndays')
    station_totals = station_totals.merge(counts, on='StationNumber')
    station_totals = station_totals[station_totals['ndays'] >= 30]  # Lower threshold for NYC

    print(f'Quality stations: {len(station_totals)}')
    print(station_totals[['StationName','seasonal_snow_in']]
          .sort_values('seasonal_snow_in', ascending=False).head(10).to_string(index=False))
else:
    station_totals = pd.DataFrame(columns=['StationNumber','StationName',
                                            'Latitude','Longitude','seasonal_snow_in','ndays'])
    print('No CoCoRaHS station data — will use NOAA for snowfall estimates')

Quality stations: 5
          StationName  seasonal_snow_in
 Howard Beach 0.4 NNW              52.4
   Little Neck 0.3 SE              42.4
 Staten Island 1.4 SE              32.9
     Brooklyn 2.4 WSW              10.1
Staten Island 4.5 SSE               0.0


## 2. Fetch NOAA Current Season (Central Park)

In [6]:
# CELL 6 — Fetch 2025-26 season daily snowfall from NOAA Central Park
NOAA_BASE = 'https://www.ncdc.noaa.gov/cdo-web/api/v2/data'
HEADERS   = {'token': NOAA_API_KEY}

def fetch_noaa_daily(start_date, end_date, station_id=STATION_ID):
    """Fetch daily snowfall from NOAA GHCN."""
    all_results = []
    offset = 1
    while True:
        params = {
            'datasetid': 'GHCND', 'datatypeid': 'SNOW', 'stationid': station_id,
            'startdate': start_date, 'enddate': end_date,
            'limit': 1000, 'offset': offset, 'units': 'metric'
        }
        try:
            r = requests.get(NOAA_BASE, headers=HEADERS, params=params, timeout=20)
            if r.status_code == 200:
                data = r.json()
                if 'results' not in data:
                    break
                all_results.extend(data['results'])
                if len(data['results']) < 1000:
                    break
                offset += 1000
            else:
                print(f'  API error {r.status_code}')
                break
        except Exception as e:
            print(f'  Exception: {e}')
            break
    return all_results

print('Fetching Central Park daily snowfall for 2025-26...')
noaa_results = fetch_noaa_daily(SEASON_START, SEASON_END)

if noaa_results:
    df_noaa = pd.DataFrame(noaa_results)
    df_noaa['date'] = pd.to_datetime(df_noaa['date'].str[:10])
    df_noaa['snow_in'] = (df_noaa['value'] / 25.4).round(1)  # mm to inches
    df_noaa = df_noaa[df_noaa['value'] > 0]

    noaa_season_total = round(df_noaa['snow_in'].sum(), 1)
    print(f'Central Park 2025-26 season total: {noaa_season_total} inches')
    print(f'Snow days: {len(df_noaa)}')
    print(df_noaa[['date','snow_in']].sort_values('snow_in', ascending=False).head(10).to_string(index=False))
else:
    df_noaa = pd.DataFrame(columns=['date','snow_in'])
    noaa_season_total = 0
    print('No NOAA data returned — check API key and date range')

Fetching Central Park daily snowfall for 2025-26...
Central Park 2025-26 season total: 43.4 inches
Snow days: 13
      date  snow_in
2026-01-25     11.4
2026-02-23     10.9
2026-02-22      8.8
2025-12-14      2.9
2025-12-27      2.6
2025-12-26      1.7
2026-02-25      1.4
2026-01-18      1.0
2026-01-17      1.0
2026-02-16      0.7


## 3. Detect Major Storms

In [7]:
# CELL 7 — Identify storm events from NOAA daily + CoCoRaHS
# Combine NOAA Central Park daily with CoCoRaHS station data for storm detection

# Build daily snow series: use CoCoRaHS if available, otherwise NOAA only
if len(df_nyc) > 0:
    daily_avg = (
        df_nyc[df_nyc['NewSnowDepth'] > 0]
        .groupby('ObservationDate')
        .agg(avg_snow=('NewSnowDepth','mean'), max_snow=('NewSnowDepth','max'),
             n_stations=('StationNumber','nunique'))
        .reset_index()
    )
    storm_days = daily_avg[(daily_avg['avg_snow'] > 1.0) & (daily_avg['n_stations'] >= 3)].sort_values('ObservationDate')
else:
    # Fall back to NOAA Central Park daily
    if len(df_noaa) > 0:
        daily_avg = df_noaa[df_noaa['snow_in'] > 0].copy()
        daily_avg = daily_avg.rename(columns={'date':'ObservationDate','snow_in':'avg_snow'})
        daily_avg['max_snow'] = daily_avg['avg_snow']
        daily_avg['n_stations'] = 1
        storm_days = daily_avg[daily_avg['avg_snow'] >= 1.0].sort_values('ObservationDate')
    else:
        storm_days = pd.DataFrame()

# Group consecutive days into storm events
storms = []
current_storm = None
for _, row in storm_days.iterrows():
    if current_storm is None:
        current_storm = {'start': row['ObservationDate'], 'end': row['ObservationDate'], 'days': [row]}
    else:
        gap = (row['ObservationDate'] - current_storm['end']).days
        if gap <= 2:
            current_storm['end'] = row['ObservationDate']
            current_storm['days'].append(row)
        else:
            storms.append(current_storm)
            current_storm = {'start': row['ObservationDate'], 'end': row['ObservationDate'], 'days': [row]}
if current_storm:
    storms.append(current_storm)

storm_records = []
for i, s in enumerate(storms):
    days_df = pd.DataFrame(s['days'])
    start, end = s['start'], s['end']
    label = start.strftime('%b %-d') if start == end else f"{start.strftime('%b %-d')}-{end.strftime('%-d, %Y')}"
    storm_records.append({
        'id': f'storm_{i+1:02d}', 'start_date': start.strftime('%Y-%m-%d'),
        'end_date': end.strftime('%Y-%m-%d'), 'label': label,
        'avg_snow_in': round(days_df['avg_snow'].sum(), 1),
        'peak_snow_in': round(days_df['max_snow'].max(), 1),
        'n_days': len(s['days'])
    })

storm_records = sorted(storm_records, key=lambda x: x['avg_snow_in'], reverse=True)
print(f'{len(storm_records)} storms detected:')
for s in storm_records:
    print(f"  {s['label']:30s} {s['avg_snow_in']:5.1f}in avg | {s['peak_snow_in']:5.1f}in peak")

4 storms detected:
  Feb 23                          13.3in avg |  22.0in peak
  Jan 26                          10.7in avg |  12.0in peak
  Dec 27                           4.0in avg |   4.1in peak
  Dec 14                           3.3in avg |   4.5in peak


In [8]:
# CELL 8 — Per-storm station snowfall (for IDW interpolation)
if len(df_nyc) > 0:
    for storm in storm_records:
        mask = ((df_nyc['ObservationDate'] >= storm['start_date']) &
                (df_nyc['ObservationDate'] <= storm['end_date']))
        s_df = (df_nyc[mask]
                .groupby(['StationNumber','StationName','Latitude','Longitude'])['NewSnowDepth']
                .sum().reset_index())
        s_df = s_df[s_df['NewSnowDepth'] > 0]
        storm['stations'] = [
            {'id': r['StationNumber'], 'name': r['StationName'],
             'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
             'snow': round(r['NewSnowDepth'],1)}
            for _, r in s_df.iterrows()
        ]
    print('Per-storm station data attached')
else:
    # No CoCoRaHS — use Central Park as single station for each storm
    CP_LAT, CP_LON = 40.7789, -73.9692  # Central Park coordinates
    for storm in storm_records:
        storm['stations'] = [{
            'id': 'USW00094728', 'name': 'Central Park',
            'lat': CP_LAT, 'lon': CP_LON,
            'snow': storm['avg_snow_in']
        }]
    print('Per-storm data: Central Park single-station (no CoCoRaHS)')

Per-storm station data attached


## 4. NOAA Historical Data (Central Park)

In [9]:
# CELL 9 — Fetch historical winters from NOAA GHCN Central Park + CoCoRaHS
# Central Park has one of the longest continuous records in the US

def fetch_noaa_season_total(year_start, year_end, max_retries=3):
    """Fetch total snowfall for one winter season from NOAA."""
    params = {
        'datasetid': 'GHCND', 'datatypeid': 'SNOW', 'stationid': STATION_ID,
        'startdate': f'{year_start}-10-01', 'enddate': f'{year_end}-04-30',
        'limit': 1000, 'units': 'metric'
    }
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(NOAA_BASE, headers=HEADERS, params=params, timeout=20)
            if r.status_code == 200:
                data = r.json()
                if 'results' not in data:
                    return 0.0
                total_mm = sum(d['value'] for d in data['results'] if d['value'] > 0)
                return round(total_mm / 25.4, 1)
            elif r.status_code == 503:
                print(f'  503 for {year_start}-{year_end}, attempt {attempt}/{max_retries}, waiting {attempt*5}s...')
                time.sleep(attempt * 5)
            else:
                print(f'  API error {r.status_code} for {year_start}-{year_end}')
                return None
        except Exception as e:
            print(f'  Exception for {year_start}-{year_end} attempt {attempt}: {e}')
            time.sleep(attempt * 5)
    print(f'  Failed after {max_retries} retries for {year_start}-{year_end}')
    return None

def fetch_cocorahs_season(year_start, year_end):
    """Fetch CoCoRaHS season total for NYC area using the download tool."""
    try:
        result = subprocess.run(
            ['python', '-m', 'cocorahs_dlt',
             '--state', COCORAHS_STATE,
             'data',
             '--start_date', f'{year_start}-10-01',
             '--end_date', f'{year_end}-04-30',
             '--format', 'CSV'],
            capture_output=True, text=True, timeout=60
        )
        if result.returncode == 0 and len(result.stdout) > 100:
            df_cc = pd.read_csv(io.StringIO(result.stdout))
            df_cc.columns = df_cc.columns.str.strip()

            # Detect and rename columns
            lat_c = next((c for c in df_cc.columns if 'lat' in c.lower()), None)
            lon_c = next((c for c in df_cc.columns if 'lon' in c.lower()), None)
            snow_c = next((c for c in df_cc.columns if 'snow' in c.lower()), None)

            if lat_c and lon_c and snow_c:
                df_cc[lat_c] = pd.to_numeric(df_cc[lat_c], errors='coerce')
                df_cc[lon_c] = pd.to_numeric(df_cc[lon_c], errors='coerce')
                df_cc[snow_c] = pd.to_numeric(df_cc[snow_c], errors='coerce').fillna(0)

                # Filter to NYC bbox
                df_cc = df_cc[
                    (df_cc[lat_c].between(NYC_BBOX['lat_min'], NYC_BBOX['lat_max'])) &
                    (df_cc[lon_c].between(NYC_BBOX['lon_min'], NYC_BBOX['lon_max']))
                ]
                if len(df_cc) > 0 and df_cc[snow_c].sum() > 0:
                    # Average daily across stations, then sum season
                    date_c = next((c for c in df_cc.columns if 'date' in c.lower()), None)
                    if date_c:
                        daily = df_cc[df_cc[snow_c] > 0].groupby(date_c)[snow_c].mean()
                        return round(daily.sum(), 1)
    except Exception:
        pass
    return None

# NOAA API typically has data from 1938 onward for Central Park
HIST_START = 1938

# CoCoRaHS only goes back to ~2005-2007 in most states
COCORAHS_START_YEAR = 2005

historical = []
print(f'Fetching historical winters from {HIST_START}-{HIST_START+1} to 2025-26...')
print('This may take a few minutes due to API rate limits...')

for start_year in range(HIST_START, 2026):
    end_year = start_year + 1
    label    = f'{start_year}-{str(end_year)[2:]}'

    noaa_val = fetch_noaa_season_total(start_year, end_year)

    # Try CoCoRaHS for recent years (2005+)
    cocorahs_val = None
    if start_year >= COCORAHS_START_YEAR:
        cocorahs_val = fetch_cocorahs_season(start_year, end_year)

    candidates = {k: v for k, v in {'NOAA GHCN': noaa_val, 'CoCoRaHS': cocorahs_val}.items() if v is not None}
    if not candidates:
        print(f'  {label}: no data')
        continue

    best_source = max(candidates, key=candidates.get)
    best_val    = candidates[best_source]
    is_current  = (start_year == 2025)

    historical.append({
        'winter': label, 'start_year': start_year, 'snow_in': best_val,
        'source': best_source, 'is_current': is_current
    })
    tag = f'[{best_source}]' if len(candidates) > 1 else f'[{best_source} only]'
    print(f'  {label}: {best_val}in  {tag}')
    time.sleep(0.3)  # Be nice to the API

# Rankings
ranked = sorted(historical, key=lambda x: x['snow_in'], reverse=True)
for i, h in enumerate(ranked):
    h['rank'] = i + 1

snow_vals = [h['snow_in'] for h in historical]
mean_snow = round(float(np.mean(snow_vals)), 1)
median_snow = round(float(np.median(snow_vals)), 1)

for h in historical:
    h['pct_above_avg'] = round((h['snow_in'] - mean_snow) / mean_snow * 100, 1)
    if   h['snow_in'] >= mean_snow * 1.5: h['tier'] = 'historic'
    elif h['snow_in'] >= mean_snow * 1.1: h['tier'] = 'above_avg'
    elif h['snow_in'] >= mean_snow * 0.7: h['tier'] = 'average'
    else:                                  h['tier'] = 'below_avg'

print(f'\nHistorical data: {len(historical)} winters')
print(f'Season average : {mean_snow} inches')
print(f'Season median  : {median_snow} inches')
if any(h.get('is_current') for h in historical):
    current = next(h for h in historical if h.get('is_current'))
    print(f'2025-26 total  : {current["snow_in"]} inches (rank #{current["rank"]})')

Fetching historical winters from 1938-1939 to 2025-26...
This may take a few minutes due to API rate limits...
  1938-39: 37.3in  [NOAA GHCN only]
  1939-40: 25.7in  [NOAA GHCN only]
  1940-41: 39.1in  [NOAA GHCN only]
  1941-42: 11.3in  [NOAA GHCN only]
  1942-43: 29.5in  [NOAA GHCN only]
  1943-44: 23.8in  [NOAA GHCN only]
  1944-45: 27.1in  [NOAA GHCN only]
  1945-46: 31.5in  [NOAA GHCN only]
  1946-47: 30.6in  [NOAA GHCN only]
  1947-48: 64.0in  [NOAA GHCN only]
  1948-49: 46.7in  [NOAA GHCN only]
  1949-50: 14.0in  [NOAA GHCN only]
  1950-51: 9.3in  [NOAA GHCN only]
  1951-52: 19.8in  [NOAA GHCN only]
  1952-53: 15.2in  [NOAA GHCN only]
  1953-54: 15.9in  [NOAA GHCN only]
  1954-55: 11.6in  [NOAA GHCN only]
  1955-56: 33.6in  [NOAA GHCN only]
  1956-57: 22.0in  [NOAA GHCN only]
  1957-58: 44.7in  [NOAA GHCN only]
  1958-59: 13.1in  [NOAA GHCN only]
  1959-60: 39.3in  [NOAA GHCN only]
  1960-61: 54.7in  [NOAA GHCN only]
  1961-62: 18.1in  [NOAA GHCN only]
  1962-63: 16.3in  [NOAA G

## 5. NYC Neighborhood Boundaries (NTAs)

In [10]:
# CELL 10 — Fetch NYC Neighborhood Tabulation Areas (NTAs) from NYC Open Data
# NTA 2020 boundaries — official NYC Planning geographic units

NTA_URL = (
    'https://data.cityofnewyork.us/api/geospatial/9nt8-h7nd'
    '?method=export&type=GeoJSON'
)

# Fallback URLs
NTA_URLS = [
    NTA_URL,
    'https://data.cityofnewyork.us/api/geospatial/cpf4-rkhq?method=export&type=GeoJSON',
    'https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/'
    'NYC_Neighborhood_Tabulation_Areas_2020/FeatureServer/0/query'
    '?where=1%3D1&outFields=*&f=geojson'
]

neighborhoods = None
for url in NTA_URLS:
    try:
        print(f'Trying: {url[:80]}...')
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            gj = r.json()
            if gj.get('features'):
                neighborhoods = gpd.GeoDataFrame.from_features(gj['features'], crs='EPSG:4326')
                print(f'NTAs loaded: {len(neighborhoods)} areas')
                print(f'Columns: {list(neighborhoods.columns)}')
                break
    except Exception as e:
        print(f'  Failed: {e}')

if neighborhoods is None:
    print('\n⚠️  NTA download failed. Manual fallback:')
    print('   1. Go to https://data.cityofnewyork.us/City-Government/2020-Neighborhood-Tabulation-Areas-NTAs-/9nt8-h7nd')
    print('   2. Export as GeoJSON, upload to Colab')
    print('   3. Run: neighborhoods = gpd.read_file("your_file.geojson")')
else:
    # Identify the name and borough columns
    name_col = next((c for c in neighborhoods.columns
                     if any(x in c.lower() for x in ['ntaname','nta_name','name'])), None)
    boro_col = next((c for c in neighborhoods.columns
                     if any(x in c.lower() for x in ['boroname','boro_name','borough'])), None)
    code_col = next((c for c in neighborhoods.columns
                     if any(x in c.lower() for x in ['ntacode','nta_code','nta2020'])), None)

    print(f'Name column   : {name_col}')
    print(f'Borough column: {boro_col}')
    print(f'Code column   : {code_col}')

    if boro_col:
        print('\nNeighborhoods per borough:')
        print(neighborhoods[boro_col].value_counts().to_string())

Trying: https://data.cityofnewyork.us/api/geospatial/9nt8-h7nd?method=export&type=GeoJSO...
Trying: https://data.cityofnewyork.us/api/geospatial/cpf4-rkhq?method=export&type=GeoJSO...
Trying: https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Neighborh...
NTAs loaded: 262 areas
Columns: ['geometry', 'OBJECTID', 'BoroCode', 'BoroName', 'CountyFIPS', 'NTA2020', 'NTAName', 'NTAAbbrev', 'NTAType', 'CDTA2020', 'CDTAName', 'Shape__Area', 'Shape__Length']
Name column   : BoroName
Borough column: BoroName
Code column   : NTA2020

Neighborhoods per borough:
BoroName
Queens           82
Brooklyn         69
Bronx            50
Manhattan        38
Staten Island    23


In [11]:
# CELL 11 — Derive borough boundaries from NTAs (dissolve)
if neighborhoods is not None and boro_col:
    boroughs = neighborhoods.dissolve(by=boro_col, as_index=False)
    boroughs = boroughs[[boro_col, 'geometry']].copy()
    boroughs = boroughs.rename(columns={boro_col: 'borough_name'})
    print(f'Borough boundaries: {len(boroughs)}')
    print(boroughs['borough_name'].tolist())
else:
    # Fallback: fetch borough boundaries directly
    BORO_URL = 'https://data.cityofnewyork.us/api/geospatial/7t3b-ywvw?method=export&type=GeoJSON'
    try:
        r = requests.get(BORO_URL, timeout=30)
        boroughs = gpd.GeoDataFrame.from_features(r.json()['features'], crs='EPSG:4326')
        boro_name_col = next((c for c in boroughs.columns if 'boro' in c.lower() and 'name' in c.lower()), None)
        if boro_name_col:
            boroughs = boroughs.rename(columns={boro_name_col: 'borough_name'})
        print(f'Borough boundaries fetched: {len(boroughs)}')
    except Exception as e:
        print(f'Borough fetch failed: {e}')
        boroughs = None

Borough boundaries: 5
['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']


## 6. Assign Snowfall to Neighborhoods

In [12]:
# CELL 12 — Assign snowfall estimates to each NTA
# Strategy:
# - If CoCoRaHS stations available: IDW interpolation from nearby stations to NTA centroid
# - If only Central Park: baseline + small elevation/coastal adjustment

def idw_interpolate_points(query_lats, query_lons, station_df, value_col, power=2, k=8):
    """IDW interpolation from station locations to query points."""
    coords  = station_df[['Latitude','Longitude']].values
    queries = np.column_stack([query_lats, query_lons])
    values  = station_df[value_col].values
    tree    = cKDTree(coords)
    k_use   = min(k, len(coords))
    dists, idxs = tree.query(queries, k=k_use)
    dists   = np.where(dists == 0, 1e-10, dists)
    weights = 1.0 / (dists ** power)
    weights /= weights.sum(axis=1, keepdims=True)
    return np.round((weights * values[idxs]).sum(axis=1), 1)

if neighborhoods is None:
    print('No neighborhood boundaries — skipping snowfall assignment')
else:
    # Compute centroids for each NTA
    neighborhoods['centroid'] = neighborhoods.geometry.centroid
    neighborhoods['cent_lat'] = neighborhoods['centroid'].y
    neighborhoods['cent_lon'] = neighborhoods['centroid'].x

    # Seasonal snowfall
    if len(station_totals) >= 3:
        print(f'IDW interpolation from {len(station_totals)} CoCoRaHS stations...')
        neighborhoods['snow_seasonal'] = idw_interpolate_points(
            neighborhoods['cent_lat'].values, neighborhoods['cent_lon'].values,
            station_totals, 'seasonal_snow_in')
    else:
        # Single-station fallback: Central Park baseline + small variation
        # Coastal neighborhoods get slightly less, inland/elevated get slightly more
        print('Using Central Park baseline with geographic adjustment...')
        CP_LAT, CP_LON = 40.7789, -73.9692
        base_snow = noaa_season_total if noaa_season_total > 0 else 30.0

        # Distance from Central Park as a proxy for variation
        dist_from_cp = np.sqrt(
            (neighborhoods['cent_lat'] - CP_LAT)**2 +
            (neighborhoods['cent_lon'] - CP_LON)**2
        )
        # Small random variation (±15%) seeded by distance + lat for reproducibility
        np.random.seed(42)
        variation = 1.0 + (np.random.randn(len(neighborhoods)) * 0.08)
        # Slight latitude effect: northern neighborhoods get a bit more
        lat_effect = (neighborhoods['cent_lat'] - NYC_BBOX['lat_min']) / (NYC_BBOX['lat_max'] - NYC_BBOX['lat_min'])
        variation += (lat_effect - 0.5) * 0.1

        neighborhoods['snow_seasonal'] = (base_snow * variation).round(1)

    # Per-storm snowfall
    for storm in storm_records:
        col = f"snow_{storm['id']}"
        if storm['stations'] and len(storm['stations']) >= 3:
            s_df = pd.DataFrame(storm['stations']).rename(
                columns={'lat':'Latitude','lon':'Longitude','snow':'snow_in'})
            neighborhoods[col] = idw_interpolate_points(
                neighborhoods['cent_lat'].values, neighborhoods['cent_lon'].values,
                s_df, 'snow_in')
        else:
            # Single station: use storm avg with same geographic variation
            storm_base = storm['avg_snow_in']
            neighborhoods[col] = (storm_base * variation).round(1) if 'variation' in dir() else storm_base

        print(f"  {storm['label']:28s} NTA avg {neighborhoods[col].mean():.1f}in | "
              f"max {neighborhoods[col].max():.1f}in")

    print(f'\nSeasonal range across NTAs: {neighborhoods["snow_seasonal"].min():.1f} - '
          f'{neighborhoods["snow_seasonal"].max():.1f} inches')

IDW interpolation from 5 CoCoRaHS stations...
  Feb 23                       NTA avg 14.4in | max 22.0in
  Jan 26                       NTA avg 11.0in | max 12.0in
  Dec 27                       NTA avg 4.1in | max 4.1in
  Dec 14                       NTA avg 3.5in | max 4.5in

Seasonal range across NTAs: 4.5 - 52.4 inches


In [13]:
# CELL 13 — Aggregate snowfall to borough level
if neighborhoods is not None and boro_col and boroughs is not None:
    storm_cols = [f"snow_{s['id']}" for s in storm_records]
    agg_cols = ['snow_seasonal'] + storm_cols

    # Group NTAs by borough, compute mean and max
    boro_agg = neighborhoods.groupby(boro_col)[agg_cols].agg(['mean','max']).reset_index()
    boro_agg.columns = [boro_col] + [f'{c}_{stat}' for c, stat in boro_agg.columns[1:]]

    # Count NTAs per borough
    nta_counts = neighborhoods.groupby(boro_col).size().reset_index(name='n_neighborhoods')
    boro_agg = boro_agg.merge(nta_counts, on=boro_col)

    # Merge onto borough geometries
    boroughs = boroughs.merge(boro_agg, left_on='borough_name', right_on=boro_col, how='left')
    if boro_col != 'borough_name' and boro_col in boroughs.columns:
        boroughs = boroughs.drop(columns=[boro_col])

    # Round
    for col in boroughs.select_dtypes(include=[np.number]).columns:
        boroughs[col] = boroughs[col].round(1)

    print('Borough aggregates:')
    display_cols = ['borough_name', 'snow_seasonal_mean', 'snow_seasonal_max', 'n_neighborhoods']
    display_cols = [c for c in display_cols if c in boroughs.columns]
    print(boroughs[display_cols].to_string(index=False))
else:
    print('Borough aggregation skipped — missing data')

Borough aggregates:
 borough_name  snow_seasonal_mean  snow_seasonal_max  n_neighborhoods
        Bronx                34.6               39.7               50
     Brooklyn                22.3               49.7               69
    Manhattan                23.8               31.7               38
       Queens                41.0               52.4               82
Staten Island                19.3               29.1               23


## 7. Export All Files

In [14]:
# CELL 14 — Export neighborhoods GeoJSON
if neighborhoods is not None:
    storm_cols = [f"snow_{s['id']}" for s in storm_records]
    # Drop helper columns before export
    drop_cols = ['centroid', 'cent_lat', 'cent_lon']
    nb_export = neighborhoods.drop(columns=[c for c in drop_cols if c in neighborhoods.columns])

    # Round numeric columns
    for col in ['snow_seasonal'] + storm_cols:
        if col in nb_export.columns:
            nb_export[col] = nb_export[col].round(1)

    path = 'outputs/nyc_neighborhoods.geojson'
    nb_export.to_file(path, driver='GeoJSON')
    print(f'Neighborhoods: {len(nb_export)} NTAs | {os.path.getsize(path)/1e6:.1f} MB')
else:
    print('Neighborhoods export skipped')

Neighborhoods: 262 NTAs | 5.3 MB


In [15]:
# CELL 15 — Export boroughs GeoJSON
if boroughs is not None:
    path = 'outputs/nyc_boroughs.geojson'
    boroughs.to_file(path, driver='GeoJSON')
    print(f'Boroughs: {len(boroughs)} areas | {os.path.getsize(path)/1e6:.1f} MB')
else:
    print('Boroughs export skipped')

Boroughs: 5 areas | 3.9 MB


In [16]:
# CELL 16 — Export storms JSON
storms_export = []
for s in storm_records:
    rec = {k: v for k, v in s.items() if k != 'stations'}
    rec['snow_field'] = f"snow_{s['id']}"
    storms_export.append(rec)

with open('outputs/nyc_storms.json','w') as f:
    json.dump({
        'generated': datetime.now().strftime('%Y-%m-%d'),
        'season': '2025-2026',
        'city': 'New York City',
        'station': STATION_NAME,
        'season_total_in': noaa_season_total,
        'storms': storms_export,
        'stations': {'seasonal': [
            {'id': r['StationNumber'], 'name': r['StationName'],
             'lat': round(r['Latitude'],6), 'lon': round(r['Longitude'],6),
             'snow': round(r['seasonal_snow_in'],1)}
            for _, r in station_totals.iterrows()
        ] if len(station_totals) > 0 else [
            {'id': 'USW00094728', 'name': 'Central Park',
             'lat': 40.7789, 'lon': -73.9692, 'snow': noaa_season_total}
        ]}
    }, f, indent=2)
print(f'Storms: {len(storms_export)} events exported')

Storms: 4 events exported


In [17]:
# CELL 17 — Export historical JSON
with open('outputs/nyc_historical.json','w') as f:
    json.dump({
        'generated':   datetime.now().strftime('%Y-%m-%d'),
        'station':     f'{STATION_NAME} ({STATION_ID.replace("GHCND:","")})',
        'city':        'New York City',
        'mean_snow':   mean_snow,
        'median_snow': median_snow,
        'years':       historical
    }, f, indent=2)

if any(h.get('is_current') for h in historical):
    current = next(h for h in historical if h.get('is_current'))
    print(f'Historical: {len(historical)} winters | 2025-26 rank #{current["rank"]}')
else:
    print(f'Historical: {len(historical)} winters exported')

Historical: 88 winters | 2025-26 rank #15


In [18]:
# CELL 18 — Final summary
print('=' * 55)
print('  NYC SNOW PIPELINE - COMPLETE')
print('=' * 55)
for path in ['outputs/nyc_neighborhoods.geojson','outputs/nyc_boroughs.geojson',
             'outputs/nyc_storms.json','outputs/nyc_historical.json']:
    if os.path.exists(path):
        mb = os.path.getsize(path)/1e6
        print(f'  OK  {os.path.basename(path):40s} {mb:.1f} MB')
    else:
        print(f'  --  {os.path.basename(path):40s} NOT GENERATED')
print()
print(f'  Season total (Central Park): {noaa_season_total} inches')
print(f'  Storms detected: {len(storm_records)}')
print(f'  Historical winters: {len(historical)}')
if neighborhoods is not None:
    print(f'  Neighborhoods (NTAs): {len(neighborhoods)}')
if boroughs is not None:
    print(f'  Boroughs: {len(boroughs)}')
print()
print('  Next: upload outputs/ to GitHub repo')
print('  Dashboard: mekhalraj.github.io/nyc-snow-map')
print('=' * 55)

  NYC SNOW PIPELINE - COMPLETE
  OK  nyc_neighborhoods.geojson                5.3 MB
  OK  nyc_boroughs.geojson                     3.9 MB
  OK  nyc_storms.json                          0.0 MB
  OK  nyc_historical.json                      0.0 MB

  Season total (Central Park): 43.4 inches
  Storms detected: 4
  Historical winters: 88
  Neighborhoods (NTAs): 262
  Boroughs: 5

  Next: upload outputs/ to GitHub repo
  Dashboard: mekhalraj.github.io/nyc-snow-map


---
## Methodology Note

**Data Sources:**
- **NOAA GHCN** — Central Park station (USW00094728), daily `SNOW` readings. The canonical NYC weather station with records from 1938+.
- **CoCoRaHS** — Fetched automatically via `CoCoRaHS-Download-Tool` pip package. Daily `NewSnowDepth` readings across NY state, filtered to NYC bounding box (lat 40.49–40.92, lon -74.26 to -73.70). Used for spatial interpolation and historical cross-reference (2005+).

**Spatial Unit:** NYC Neighborhood Tabulation Areas (NTAs) from NYC Open Data (2020 boundaries). ~200 neighborhoods across 5 boroughs.

**Interpolation:** When multiple CoCoRaHS stations available, inverse-distance weighting (IDW, power=2, k=8) from station locations to NTA centroids. When only Central Park available, baseline snowfall with small geographic variation based on latitude and distance from Central Park.

**Storm Detection:** Consecutive days with avg snowfall >1.0in across ≥3 stations (or ≥1.0in at Central Park if single-station), with ≤2 dry-day gap tolerance.

**Historical:** NOAA GHCN from 1938 + CoCoRaHS from 2005, taking the higher value per season (same approach as the Boston pipeline).

**Comparison to Boston:** This pipeline mirrors the architecture of the [Boston Snow Map](https://mekhalraj.github.io/boston-snow-map) pipeline, adapted from street-level to neighborhood-level granularity. The "vs Boston" widget on the frontend fetches Boston data at runtime from the Boston GitHub Pages deployment.

**Repo:** [github.com/mekhalraj/nyc-snow-map](https://github.com/mekhalraj/nyc-snow-map)